In [1]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/ndainoran/Desktop/PORTAL/deidentification/deIdentification/')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task
from qc_package.scanner import DbScanner
import pandas as pd

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [2]:
qc_config = {
    "PATIENT_ID": {"prefix_value": "1001000", "length_of_value": 15},
    "ENCOUNTER_ID": {"prefix_value": "1001000", "length_of_value": 19},
}
db = DbDetailsModel.objects.get(db_name="noren_prod_local")

db_scanner = DbScanner(
    db.source_db_config['connection_str'],
    db.destination_db_config['connection_str'],
    db.run_config["mapping_db_config"],
    qc_config
)

In [2]:
code_failure = []
import traceback
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    # if table_obj.table_name in tables_for_qc:
    try:
        table_config = table_obj.table_details_for_ui
        output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
        table_obj.qc_result = output_result
        table_obj.save()
    except Exception as e:
        error = traceback.format_exc()
        tablename = table_obj.table_name
        code_failure.append((error, tablename))

In [1]:
len(code_failure)

NameError: name 'code_failure' is not defined

In [11]:
with open('code_failure_qc.txt', 'w') as fp:
    fp.write(str(code_failure))

In [5]:
import os, json
# table_name, source_row_count, dest_row_count, qc_status, ignore_rows_count, reason, is_phi
all_results = []

def is_phi_table(table_config):
    for col_conf in table_config['columns_details']:
        if col_conf['is_phi']:
            return True
    return False
    
file_path = '/NOTEBOOK/Noran/Auto_qct'
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    qc_result = table_obj.qc_result
    if not qc_result:
        continue
    one_table = {
        "table_name": table_obj.table_name,
        "source_rows_count": qc_result['source_rows_count'],
        "dest_rows_count": qc_result['dest_rows_count'],
        "qc_status": qc_result.get("final_qc_result", {}).get("is_qc_passed"),
        "ignore_rows_count": qc_result['ignore_rows_count'],
        "reason": qc_result.get("final_qc_result", {}).get("reason"),
        "is_phi": is_phi_table(table_obj.table_details_for_ui)
    }
    all_results.append(one_table)
len(all_results)
res_df = pd.DataFrame(all_results)
res_df.shape
# NOTEBOOK/Noran/Auto_qc
res_df.to_csv("Auto_qc/qc_results_03062025.csv", index=False)

In [3]:
code_failure = []
import traceback
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    if table_obj.qc_result  in [{}, None]:
        code_failure.append(table_obj.table_name)
    # # if table_obj.table_name in tables_for_qc:
    # try:
    #     table_config = table_obj.table_details_for_ui
    #     output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
    #     table_obj.qc_result = output_result
    #     table_obj.save()
    # except Exception as e:
    #     error = traceback.format_exc()
    #     tablename = table_obj.table_name
    #     code_failure.append((error, tablename))

In [4]:
len(code_failure)

119

In [ ]:
# code_failure = []
d = []
f = []
import traceback
from tqdm import tqdm
table_objs = TableDetailsModel.objects.filter(table_name__in=code_failure).order_by('rows_count')
for table_obj in tqdm(table_objs, "running"):
    # if table_obj.qc_result  in [{}, None]:
    #     code_f ailure.append(table_obj.table_name)
    # # if table_obj.table_name in tables_for_qc:
    try:
        table_config = table_obj.table_details_for_ui
        output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
        table_obj.qc_result = output_result
        table_obj.save()
        d.append(table_obj.table_name)
    except Exception as e:
        error = traceback.format_exc()
        tablename = table_obj.table_name
        f.append((error, tablename))

running:  96%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎      | 114/119 [1:17:37<37:00, 444.01s/it]

In [6]:
f[0]

('Traceback (most recent call last):\n  File "/var/folders/c5/vhmk0_vd5kz1ts8y6rk0zbwh0000gn/T/ipykernel_71122/4114150219.py", line 12, in <module>\n    output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)\nNameError: name \'db_scanner\' is not defined\n',
 'deletehistory')